# Statistical Analysis for TSB-AD

## Imports

In [ ]:
import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
import networkx
import numpy as np
import pandas as pd
from scikit_posthocs import posthoc_nemenyi_friedman
from scipy.stats import friedmanchisquare, pearsonr, spearmanr
import seaborn as sns

# Style settings for plots
sns.set_theme(style='whitegrid', context='paper', font_scale=1.2, font='serif', palette='Set2')
# Use a serif font globally
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']

results_dir = Path('eval')
plots_dir = Path('eval/plots')
plots_dir.mkdir(parents=True, exist_ok=True)

# TSB-AD leaderboard models missing from the benchmark_eval_results
leaderboard_files = [os.path.basename(f) for f in Path('leaderboard_results/').glob('*.csv')]
missing_models_uni = ['CHARM', 'MMPAD', 'Time-RCD', 'TSPulse (FT)', 'TSPulse (ZS)', 'xLSTMAD']
missing_models_multi = ['CHARM', 'MMPAD', 'TSPulse (FT)', 'TSPulse (ZS)', 'xLSTMAD']

# Models added in this PhD research
NEW_MODELS = ['PyCaret-IForest', 'PyOD-IForest', 'LUNAR']
FILE_PATHS = [
    (NEW_MODELS[0], 'PyCaretADModel.csv'),
    (NEW_MODELS[1], 'TimeSeriesODModel.csv'),
    (NEW_MODELS[2], 'LunarADModel.csv')
]

# Univariate models recorded in benchmark_eval_results
BENCHMARK_MODELS_UNI = [
    'Sub-IForest', 'IForest', 'Sub-LOF', 'LOF', 'POLY', 'MatrixProfile', 'KShapeAD', 'SAND',
    'Series2Graph', 'SR', 'Sub-PCA', 'Sub-HBOS', 'Sub-OCSVM', 'Sub-MCD', 'Sub-KNN', 'KMeansAD',
    'AutoEncoder', 'CNN', 'LSTMAD', 'TranAD', 'AnomalyTransformer', 'OmniAnomaly', 'USAD', 'Donut',
    'TimesNet', 'FITS', 'OFA', 'Lag-Llama', 'Chronos', 'TimesFM', 'MOMENT (ZS)', 'MOMENT (FT)'
]
BENCHMARK_MODELS_MULTI = [
    'IForest', 'LOF', 'PCA', 'HBOS', 'OCSVM', 'MCD', 'KNN', 'KMeansAD', 'COPOD', 'CBLOF', 'EIF',
    'RobustPCA', 'AutoEncoder', 'CNN', 'LSTMAD', 'TranAD', 'AnomalyTransformer', 'OmniAnomaly',
    'USAD', 'Donut', 'TimesNet', 'FITS', 'OFA'
]

# Combine the benchmark models, new models, and missing leaderboard models
MODELS_UNIVARIATE = [*BENCHMARK_MODELS_UNI, *NEW_MODELS, *missing_models_uni]
MODELS_MULTIVARIATE = [*BENCHMARK_MODELS_MULTI, *NEW_MODELS, *missing_models_multi]

## Datasets and Models

In [ ]:
# DATASETS
dataset_files = {
    'All Univariate Files': 'TSB-AD-U.csv',
    'Univariate (HPO)': 'TSB-AD-U-Tuning.csv',
    'Univariate (smaller test set)': 'TSB-AD-U-Eva.csv',
    'Univariate (full test set)': 'TSB-AD-U-Eva-Full.csv',
    'All Multivariate Files': 'TSB-AD-M.csv',
    'Multivariate (HPO)': 'TSB-AD-M-Tuning.csv',
    'Multivariate (test set)': 'TSB-AD-M-Eva.csv',
}
df_dataset_counts = pd.DataFrame({
    'Dataset': list(dataset_files.keys()),
    'Source': list(dataset_files.values()),
    'Count': [len(pd.read_csv(f'../Datasets/File_List/{f}')) for f in dataset_files.values()]
})
# display(df_dataset_counts)
# Save as tex file
df_dataset_counts.to_latex(
    'eval/dataset_counts.tex', index=False, escape=True,
    caption='Counts of datasets used in the TSB-AD benchmark evaluation.',
    label='tab:ch5:dataset_counts',
    column_format='p{6.0cm}p{5.5cm}m{1.5cm}',
)
# Add horizontal line after univariate rows
with open('eval/dataset_counts.tex', 'r', encoding='utf-8') as f:
    lines = f.readlines()
for i, line in enumerate(lines):
    if 'Eva-Full' in line:
        lines.insert(i + 1, '\\hline\n')
with open('eval/dataset_counts.tex', 'w', encoding='utf-8') as f:
    f.writelines(lines)

In [ ]:
# MODEL RESULTS
data = {
    'Source': [
        'TSB-AD benchmark (U)', 'TSB-AD leaderboard (U)', 'This research', 'Total univariate models',
        'TSB-AD benchmark (M)', 'TSB-AD leaderboard (M)', 'This research', 'Total multivariate models',
    ],
    'Location': [
        'uni_mergedTable_VUS-PR.csv', 'leaderboard_results folder', '-', '-',
        'multi_mergedTable_VUS-PR.csv', 'leaderboard_results folder', '-', '-',
    ],
    'Count': [
        len(BENCHMARK_MODELS_UNI), len(missing_models_uni), len(NEW_MODELS), len(MODELS_UNIVARIATE),
        len(BENCHMARK_MODELS_MULTI), len(missing_models_multi), len(NEW_MODELS), len(MODELS_MULTIVARIATE)
    ]
}
df_model_counts = pd.DataFrame(data)
display(df_model_counts)

# Save the model names to a LaTeX file
df_model_counts.to_latex(
    'eval/model_counts.tex', index=False, escape=True,
    caption='Counts of univariate and multivariate time series anomaly detection models.',
    label='tab:ch5:model_counts', column_format='p{4.8cm}p{6.5cm}m{1.5cm}',
)

# Add horizontal lines after the two "Total" rows
with open('eval/model_counts.tex', 'r', encoding='utf-8') as f:
    lines = f.readlines()
for i, line in enumerate(lines):
    if 'Total univariate' in line:
        lines.insert(i + 1, '\\hline\n')
with open('eval/model_counts.tex', 'w', encoding='utf-8') as f:
    f.writelines(lines)

In [ ]:
df_model_names = pd.DataFrame({
    'Source': [
        'TSB-AD benchmark (U)', 'TSB-AD leaderboard (U)', 'TSB-AD benchmark (M)',
        'TSB-AD leaderboard (M)', 'This research (U and M)'
    ],
    'Models': [
        ', '.join(BENCHMARK_MODELS_UNI),
        ', '.join(missing_models_uni),
        ', '.join(BENCHMARK_MODELS_MULTI),
        ', '.join(missing_models_multi),
        ', '.join(NEW_MODELS)
    ]
})
display(df_model_names)
# Save the model names to a LaTeX file
df_model_names.to_latex(
    'eval/model_names.tex', index=False, escape=True,
    caption='Names of univariate and multivariate time series anomaly detection models.',
    label='tab:ch5:model_names',
    column_format='p{2.65cm}p{10.9cm}',
)
# Add horizontal lines
with open('eval/model_names.tex', 'r', encoding='utf-8') as f:
    lines = f.readlines()
for i, line in enumerate(lines):
    if 'TSB-AD leaderboard' in line or 'TSB-AD benchmark' in line:
        lines.insert(i + 1, '\\hline\n')

# Set font size
lines.insert(3, '\\small\n')

with open('eval/model_names.tex', 'w', encoding='utf-8') as f:
    f.writelines(lines)

## Check Data

In [ ]:
# Check to see if the models results reported in Uni_TSPulse.csv is consistent with the benchmark_eval_results
df_tspulse = pd.read_csv('leaderboard_results/Uni_TSPulse.csv')
benchmark_results = pd.read_csv('benchmark_eval_results/uni_mergedTable_VUS-PR.csv')

for col in df_tspulse.columns:
    if col in MODELS_UNIVARIATE:
        if col not in benchmark_results.columns or col not in df_tspulse.columns:
            continue
        benchmark_scores = benchmark_results[col]
        df_tspulse_scores = df_tspulse[col]
        try:
            assert benchmark_scores.round(4).values.flatten().tolist() == df_tspulse_scores.round(4).values.flatten().tolist(), f"Scores for {col} do not match between benchmark and TSPulse results."
        except AssertionError as e:
            print(e)
            print(benchmark_scores)
            print(df_tspulse_scores)
            print(sum(benchmark_scores), sum(df_tspulse_scores))

## Inserting New Results Into Benchmark Results

In [ ]:
# Read the existing results and the new PyCaret results
def add_new_results(mode: str = 'uni'):
    # Read benchmark results
    df_existing_results = pd.read_csv(f'benchmark_eval_results/{mode}_mergedTable_VUS-PR.csv')

    # Read results of missing leaderboard models
    path_prefix = f'leaderboard_results/{mode.capitalize()}_'
    df_existing_results['CHARM'] = pd.read_csv(f'{path_prefix}CHARM.csv')['CHARM']
    df_existing_results['MMPAD'] = pd.read_csv(f'{path_prefix}MMPAD.csv')['VUS-PR']
    df_existing_results['xLSTMAD'] = pd.read_csv(f'{path_prefix}xLSTMAD.csv')['VUS-PR']
    df_tspulse = pd.read_csv(f'{path_prefix}TSPulse.csv')
    df_existing_results['TSPulse (FT)'] = df_tspulse['TSPulse (FT)']
    df_existing_results['TSPulse (ZS)'] = df_tspulse['TSPulse (ZS)']
    if mode == 'uni':
        df_existing_results['Time-RCD'] = pd.read_csv(f'{path_prefix}Time_RCD.csv')['VUS-PR']

    # Read the results of the new models
    for model_name, file_name in FILE_PATHS:
        df_model = pd.read_csv(f'{results_dir}/metrics/{mode}/{file_name}')
        df_existing_results[model_name] = df_model['VUS-PR']
        # Ensure that the 'HP' column only has one value in each DataFrame
        assert df_model['HP'].nunique() == 1

    # Move new results columns to second position (after the first column)
    cols = df_existing_results.columns.tolist()
    df_existing_results = df_existing_results[[cols[0]] + [cols[-1], cols[-2]] + cols[1:-2]]

    # Save the updated DataFrame to a new CSV file
    df_existing_results.to_csv(f'{results_dir}/{mode}_expanded_VUS-PR.csv', index=False)

    # Print number of missing values in the new columns
    for col in df_existing_results.columns:
        missing_count = df_existing_results[col].isna().sum()
        if missing_count > 0:
            print(f"Column '{col}' has {missing_count} missing values out of {len(df_existing_results)} rows.")

add_new_results('uni')
add_new_results('multi')

## Evaluate HPO Results

In [ ]:
# %%capture
# Evaluate HPO Results
def evaluate_hpo_results(mode: str, model: str, file_name: str):
    """Generate box plots of VUS-PR values for each algorithm across all files.

    :param str mode: 'uni' for univariate or 'multi' for multivariate
    :param str model: Model name (e.g., 'PyCaret', 'PyOD', 'LUNAR')
    :param str file_name: Name of CSV file containing HPO results (e.g., 'PyCaretADModel.csv', 'TimeSeriesODModel.csv', 'LunarADModel.csv')
    """
    df_results = pd.read_csv(f'{results_dir}/HP_tuning/{mode}/{file_name}')
    print(df_results.shape)

    # DataFrame of files (index) by algorithms (columns) with VUS-PR values
    pivot_table = df_results.pivot(index='file', columns='HP', values='VUS-PR')

    # Sort columns by average performance
    avg_performance = pivot_table.median().sort_values(ascending=False)
    pivot_table = pivot_table[avg_performance.index]

    # Rename columns
    readable = [c.replace("{'algorithm': '", "").replace("'}", "").upper() for c in pivot_table.columns]
    pivot_table.columns = readable

    # Boxplot of VUS-PR values for each algorithm across all files sorted by median performance
    width = 4 + (len(pivot_table.columns) * 0.25)  # Adjust width based on number of algorithms
    height = 2 + (width * 0.5)  # Adjust height based on width for better aspect ratio
    plt.figure(figsize=(width, height))
    sns.boxplot(data=pivot_table, showfliers=False,  meanprops=dict(color='k', linestyle='--'), showmeans=True, meanline=True)
    plt.xticks(rotation=45, ha='right')
    print(f'VUS-PR Distribution for {model} HPO Results ({mode.capitalize()})')
    # plt.title(f'VUS-PR Distribution for {model} HPO Results ({mode.capitalize()})', fontsize=12)
    plt.tight_layout()
    plt.grid()
    plt.savefig(f'{results_dir}/HP_tuning/{mode}/{model}_HPO_VUS-PR_boxplot.png', dpi=300)
    plt.show()

# for mode in ['uni', 'multi']:
#     for model, file_name in FILE_PATHS:
#         try:
#             evaluate_hpo_results(mode, model, file_name)
#         except Exception as e:
#             print(f'Error evaluating HPO results for {model} ({mode}): {e}')

## TSB-AD-U

In [ ]:
def read_model_scores(results_dir: str, mode: str):
    # df_VUS_PR = pd.read_csv('benchmark_eval_results/uni_mergedTable_VUS-PR.csv')  # Original results
    df_VUS_PR = pd.read_csv(f'{results_dir}/{mode}_expanded_VUS-PR.csv')  # Updated results with PyCaret

    if mode == 'uni':
        model_names = MODELS_UNIVARIATE
    elif mode == 'multi':
        model_names = MODELS_MULTIVARIATE
    else:
        raise ValueError("Invalid mode. Use 'uni' for univariate or 'multi' for multivariate.")

    # Calculate the mean VUS-PR for each solution and rank them
    df_mean = pd.DataFrame()
    df_mean['VUS_PR_Rank'] = df_VUS_PR[model_names].mean().rank(ascending=False)
    print('Dropped:', set(df_VUS_PR.columns) - set(model_names))

    # Sort ranks
    df_sorted_ranks = df_mean.sort_values(by='VUS_PR_Rank', ascending=True)
    # print(df_sorted_ranks)

    return df_sorted_ranks, df_VUS_PR, model_names

df_uni_sorted_ranks, df_VUS_PR_uni, models_uni = read_model_scores('eval', 'uni')

In [ ]:
# %%capture
def boxplot_model_scores(mode: str, df_VUS_PR: pd.DataFrame, df_sorted_ranks: pd.DataFrame, figsize=(8, 8)):
    """Generate box plot of model VUS-PR scores

    :param str mode: Type of time series data (e.g., 'uni' for univariate, 'multi' for multivariate)
    :param pd.DataFrame df_VUS_PR: VUS-PR scores for each model across all files
    :param pd.DataFrame df_sorted_mean: DataFrame containing the mean VUS-PR scores and ranks for each model
    :param tuple figsize: Size of the figure (width, height)
    """
    rank_list = df_sorted_ranks.index  # [:12]  # Original

    highlight = {
        'PyCaret-IForest': 'red', 'PyOD-IForest': 'red',
        'LUNAR': 'yellow',
        'IForest': 'orange',
    }
    palette = [highlight.get(col, 'lightblue') for col in df_VUS_PR[rank_list].columns]

    plt.figure(figsize=figsize)
    ax = sns.boxplot(
        data=df_VUS_PR[rank_list], showfliers=False, meanprops=dict(color='k', linestyle='--'),
        showmeans=True, meanline=True, orient='h', palette=palette,
    )
    # Model names. Set highlighted models to be bold
    plt.yticks(ticks=range(len(rank_list)), labels=rank_list, fontsize=12)
    for label in ax.get_yticklabels():
        if label.get_text() in highlight:
            label.set_fontweight('bold')

    plt.xlabel('VUS-PR', fontsize=12)

    # Make the box outlines black for the highlighted models
    for i, patch in enumerate(ax.patches):
        if rank_list[i] in highlight:
            patch.set_edgecolor('black')
            patch.set_linewidth(1)
            # Whiskers, caps, median, etc.
            for line in ax.lines[i * 6:(i + 1) * 6]:
                line.set_color('black')
                line.set_linewidth(1)

    plt.tight_layout()
    plt.savefig(f'{plots_dir}/{mode}_VUS-PR_boxplot.png', dpi=500)
    plt.show()

# Box plot of VUS-PR scores for univariate models
boxplot_model_scores('uni', df_VUS_PR_uni, df_uni_sorted_ranks, figsize=(8, 8))

## TSB-AD-M

In [ ]:
df_multi_sorted_ranks, df_VUS_PR_multi, models_multi = read_model_scores('eval', 'multi')

In [ ]:
# %%capture

# Box plot of VUS-PR scores for multivariate models
boxplot_model_scores('multi', df_VUS_PR_multi, df_multi_sorted_ranks)

## CD Diagrams

In [ ]:
def Friedman_Nemenyi(df_perf: pd.DataFrame, alpha=0.01):
    df_counts = pd.DataFrame({'count': df_perf.groupby(['classifier_name']).size()}).reset_index()
    # Record the maximum number of datasets
    max_nb_datasets = df_counts['count'].max()
    # Create a list of classifiers
    classifiers = list(df_counts.loc[df_counts['count'] == max_nb_datasets]['classifier_name'])

    # Compute friedman p-value
    friedman_p_value = friedmanchisquare(
        *(np.array(df_perf.loc[df_perf['classifier_name'] == c]['accuracy']) for c in classifiers)
    )[1]

    # Decide whether to reject the null hypothesis
    # If p-value >= alpha: we cannot reject the null hypothesis. No statistical difference.
    if friedman_p_value >= alpha:
        print('No statistical difference...')
        raise Exception()
        return None, None, None
    # Friedman test OK
    # Prepare input for Nemenyi test
    data = []
    for c in classifiers:
        data.append(df_perf.loc[df_perf['classifier_name'] == c]['accuracy'])
    data = np.array(data, dtype=np.float64)

    # Conduct the Nemenyi post-hoc test
    # print(classifiers)
    # Order is classifiers' order
    nemenyi = posthoc_nemenyi_friedman(data.T)

    # Original code: p_values.append((classifier_1, classifier_2, p_value, False)), True: represents there exists statistical difference
    p_values = []

    # Comparing p-values with the alpha value
    for nemenyi_index in nemenyi.index:
        for nemenyi_columns in nemenyi.columns:
            if nemenyi_index < nemenyi_columns:
                # Get the model names corresponding to the indices
                model_1, model_2 = classifiers[nemenyi_index], classifiers[nemenyi_columns]  # type: ignore
                # Compare the p-value with the alpha value and append to p_values
                score = nemenyi.loc[nemenyi_index, nemenyi_columns]
                result = score < alpha
                p_values.append((model_1, model_2, score, result))
            else:
                continue

    # Nemenyi test OK

    m = len(classifiers)

    # Sort by classifier name then by dataset name
    sorted_df_perf = df_perf.loc[df_perf['classifier_name'].isin(classifiers)]. \
        sort_values(['classifier_name', 'dataset_name'])

    rank_data = np.array(sorted_df_perf['accuracy']).reshape(m, max_nb_datasets)

    df_ranks = pd.DataFrame(data=rank_data, index=np.sort(classifiers), columns=np.unique(sorted_df_perf['dataset_name']))

    # dfff = df_ranks.rank(ascending=False)
    # compute average rank
    average_ranks = df_ranks.rank(ascending=False).mean(axis=1).sort_values(ascending=False)

    return p_values, average_ranks, max_nb_datasets

def graph_ranks(
    average_ranks, names, p_values, cd=None, cdmethod=None, low_value=None, high_value=None,
    width=11, textspace=1, reverse=False, filename=None, **kwargs
):
    textspace = float(textspace)

    # n th column
    def nth(l: list, n: int):
        n = lloc(l, n)
        return [a[n] for a in l]

    # Get the location of an element in a list, counting from front or back
    def lloc(l: list, n: int):
        """Return an integer, count from front or from back."""
        if n < 0:
            return len(l[0]) + n
        else:
            return n

    if low_value is None:
        low_value = min(1, int(math.floor(min(average_ranks))))
    if high_value is None:
        high_value = max(len(average_ranks), int(math.ceil(max(average_ranks))))

    cline = 0.4
    rank_count = len(average_ranks)
    lines_blank = 0
    scale_width = width - 2 * textspace

    # Position of rank
    def rank_pos(rank):
        if not reverse:
            a = rank - low_value
        else:
            a = high_value - rank
        # Set up the format
        return textspace + scale_width / (high_value - low_value) * a

    # Add a small offset to the cline to avoid overlapping with the top line
    cline += 0.25

    # set up the formats
    min_insignificant = max(2 * 0.2, lines_blank)
    height = cline + ((rank_count + 1) / 2) * 0.2 + min_insignificant + 2

    # matplotlib figure format setup
    fig = plt.figure(figsize=(width, height))
    fig.set_facecolor('white')
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_axis_off()

    height_fraction = 1. / height
    width_fraction = 1. / width

    def scale_to_height(coords):
        """Convert a list of values to figure coordinates based on height."""
        return [a * height_fraction for a in coords]

    def scale_to_width(coords):
        """Convert a list of values to figure coordinates based on width."""
        return [a * width_fraction for a in coords]

    ax.plot([0, 1], [0, 1], c="w")
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)

    # Line plots
    def line(coords, color='k', **kwargs):
        """Draw a line on the plot."""
        ax.plot(scale_to_width(nth(coords, 0)), scale_to_height(nth(coords, 1)), color=color, **kwargs)

    # Add text to the plot
    def text(x, y, s, *args, **kwargs):
        ax.text(width_fraction * x, height_fraction * y, s, *args, **kwargs)

    # TOP LINE
    line([(textspace, cline), (width - textspace, cline)], linewidth=0.7)

    big_tick, small_tick = 0.1, 0.05
    # linewidth, linewidth_sign = 2.0, 4.0  # ORIGINAL
    linewidth, linewidth_sign = 1.0, 2.0  # UPDATED

    tick = None

    # [low_value, high_value], step size is 0.5
    for a in list(np.arange(low_value, high_value, 0.5)) + [high_value]:
        tick = small_tick
        if a == int(a):
            tick = big_tick
        # Plot a line for each rank position
        line([(rank_pos(a), cline - tick / 2), (rank_pos(a), cline)], linewidth=0.7)

    # Add text to the plot, only for integer value
    for a in range(low_value, high_value + 1):
        text(rank_pos(a), cline - tick / 2 - 0.05, str(a), ha="center", va="bottom", size=16)

    rank_count = len(average_ranks)
    space_between_names = 0.24

    # Format for the first half of algorithms
    for i in range(math.ceil(rank_count / 2)):
        current_height = cline + min_insignificant + i * space_between_names
        rank_position = (rank_pos(average_ranks[i]), cline)
        rank_marker = (rank_pos(average_ranks[i]), current_height)
        text_position = (textspace - 0.1, current_height)

        # Plot a line from the rank position to the text position
        line([rank_position, rank_marker, text_position], linewidth=linewidth)

        # Add text for the algorithm name, aligned to the right
        # If the model name in NEW_MODELS, make it bold and red
        if names[i] in NEW_MODELS:
            color = 'red'
            fontweight = 'bold'
        else:
            color = 'k'
            fontweight = 'normal'
        text(
            textspace - 0.2, current_height, names[i], color=color,
            ha="right", va="center", size=16, fontweight=fontweight
        )

    # Format for the second half of algorithms
    for i in range(math.ceil(rank_count / 2), rank_count):
        current_height = cline + min_insignificant + (rank_count - i - 1) * space_between_names

        # Plot a line from the rank position to the text position
        points = [
            (rank_pos(average_ranks[i]), cline),
            (rank_pos(average_ranks[i]), current_height),
            (textspace + scale_width + 0.1, current_height)
        ]
        line(points, linewidth=linewidth)

        # Add text for the algorithm name, aligned to the left
        if names[i] in NEW_MODELS:
            color = 'red'
            fontweight = 'bold'
        else:
            color = 'k'
            fontweight = 'normal'
        text(
            textspace + scale_width + 0.2, current_height, names[i], color=color,
            ha="left", va="center", size=16, fontweight=fontweight
        )

    # Generate cliques and plot a line to connect elements in cliques
    start = cline + 0.2
    side = -0.02
    height = 0.1
    cliques = form_cliques(p_values, names)
    achieved_half = False

    # Plot a line to connect elements in cliques
    for clq in cliques:
        if len(clq) == 1:
            continue
        # Get the minimum and maximum indices of the clique
        min_idx = np.array(clq).min()
        max_idx = np.array(clq).max()
        if min_idx >= len(names) / 2 and achieved_half is False:
            start = cline + 0.25
            achieved_half = True
        # Draw a line connecting the minimum and maximum indices of the clique
        points = [(rank_pos(average_ranks[min_idx]) - side, start), (rank_pos(average_ranks[max_idx]) + side, start)]
        line(points, color='darkred', linewidth=linewidth_sign)
        start += height

def form_cliques(p_values: list, names: list):
    """Form cliques based on p-values from the Nemenyi test.

    :param list p_values: List of tuples containing (classifier_1, classifier_2, p_value, significant_difference)
    :param list names: List of classifier names
    :return: Generator of maximal cliques in the undirected graph formed by the classifiers
    """
    g_data = np.zeros((len(names), len(names)), dtype=np.int64)
    for p in p_values:
        if p[3] is False:
            i = np.where(names == p[0])[0][0]
            j = np.where(names == p[1])[0][0]
            min_i = min(i, j)
            max_j = max(i, j)
            g_data[min_i, max_j] = 1
    g = networkx.Graph(g_data)
    # Returns all maximal cliques in an undirected graph.
    return networkx.find_cliques(g)

In [ ]:
def plot_cd_diagram(
    df_scores: pd.DataFrame,
    mode: str = 'uni',
    anomaly_type: str = 'point_anomaly',
    width: int = 10,
    textspace: float = 1.4
):
    """Plot Critical Difference diagram.

    :param pd.DataFrame df_scores: DataFrame containing the scores for each model and dataset
    :param str mode: Type of time series data, defaults to 'uni'
    :param str anomaly_type: Type of anomaly, defaults to 'point_anomaly'
    :param int width: Width of the diagram, defaults to 10
    :param float textspace: Space for text in the diagram, defaults to 1.2
    """
    # model_names = ['Sub-PCA', 'MOMENT (FT)', 'MOMENT (ZS)', 'POLY', 'CNN', 'SR', 'Series2Graph', 'LSTMAD', 'IForest', 'USAD']
    if mode == 'uni':
        model_names = models_uni
    else:
        model_names = models_multi
    model_names = [model for model in model_names if model in df_scores.columns]

    df_filter = df_scores[df_scores[anomaly_type] == 1]
    # df_filter = df_scores[df_scores['point_anomaly'] == 1]  # Point anomalies
    # # df_filter = df_scores[df_scores['seq_anomaly'] == 1]  # Sequence anomalies
    # # df_filter = df_scores[df_scores['num_anomaly'] == 1]  # Single anomaly
    # # df_filter = df_scores[df_scores['num_anomaly'] > 1]  # Multiple anomalies

    eval_list = []
    for _, row in df_filter.iterrows():
        for method in model_names:
            eval_list.append([method, row['file'], row[method]])
    eval_df = pd.DataFrame(eval_list, columns=['classifier_name', 'dataset_name', 'accuracy'])
    p_values, average_ranks, _ = Friedman_Nemenyi(df_perf=eval_df, alpha=0.01)

    graph_ranks(
        average_ranks.values, average_ranks.keys(), p_values,
        cd=None, reverse=True, width=width, textspace=textspace,
    )
    # plt.title(f'Critical Difference Diagram ("{mode}", {anomaly_type}, ' + r'$\alpha$' + '=0.01)', fontsize=15)
    # plt.tight_layout()
    plt.savefig(f'{plots_dir}/CD_{mode}_{anomaly_type}.png', dpi=500, bbox_inches='tight')
    plt.show()

df_scores_uni = pd.read_csv(f'{results_dir}/uni_expanded_VUS-PR.csv')
plot_cd_diagram(df_scores_uni, 'uni', anomaly_type='point_anomaly', width=11, textspace=1)
plot_cd_diagram(df_scores_uni, 'uni', anomaly_type='seq_anomaly', width=11, textspace=1)

In [ ]:
df_scores_multi = pd.read_csv(f'{results_dir}/multi_expanded_VUS-PR.csv')
plot_cd_diagram(df_scores_multi, 'multi', anomaly_type='point_anomaly', width=11, textspace=1)
plot_cd_diagram(df_scores_multi, 'multi', anomaly_type='seq_anomaly', width=11, textspace=1)

# Combined Results

In [ ]:
def save_combined_df(df_combined: pd.DataFrame, name: str, rounding: int = 4, font_size: str = 'small') -> pd.DataFrame:
    """Save the combined DataFrame to CSV and LaTeX files, and format new models in bold.

    :param pd.DataFrame df_combined: DataFrame containing combined mean scores or ranks
    :param str name: Name. i.e. 'scores' or 'ranks'
    :param int rounding: Number of decimal places to round the values
    :return pd.DataFrame: The modified DataFrame
    """
    # Save to CSV
    df_combined.to_csv('eval/combined_mean_scores.csv')
    # Fill NaN values with '-'
    df_combined = df_combined.fillna('-')

    # Make rows containing the new models bold in the LaTeX table
    def _bold_new_models(row):
        if row.name in NEW_MODELS + ['IForest']:
            return ['\\textbf{' + str(round(x, rounding)) + '}' for x in row]
        else:
            return row

    df_combined = df_combined.apply(_bold_new_models, axis=1)

    # Make the index bold for new models
    df_combined.index = df_combined.index.map(
        lambda x: '\\textbf{' + str(x) + '}' if x in NEW_MODELS + ['IForest'] else str(x)
    )

    # Save to LaTeX
    tex_path = f'eval/combined_mean_{name}.tex'
    df_combined.to_latex(
        tex_path, index=True, header=True, float_format=f"%.{rounding}f",
        caption=f'Mean VUS-PR {name.capitalize()} for univariate and multivariate models',
        label=f'tab:ch5:combined_mean_{name}',
        column_format='lcc',  # column_format='lp{3.0cm}p{3.0cm}',
    )

    # Set font size
    with open(tex_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    lines.insert(3, f'\\centering \\{font_size}\n')
    with open(tex_path, 'w', encoding='utf-8') as f:
        f.writelines(lines)

    # return df_combined

# Combine mean VUS-PR scores for each model into a single DataFrame
df_combined_scores = pd.DataFrame({
    'Mean VUS-PR (Uni.)': df_VUS_PR_uni[models_uni].mean().sort_values(ascending=False),
    'Mean VUS-PR (Multi.)': df_VUS_PR_multi[models_multi].mean().sort_values(ascending=False)
}).sort_values(by='Mean VUS-PR (Uni.)', ascending=False)
save_combined_df(df_combined_scores, 'scores', rounding=4, font_size='footnotesize')

# Same but using mean ranks
df_combined_ranks = pd.DataFrame({
    'VUS-PR Rank (Uni.)': df_uni_sorted_ranks['VUS_PR_Rank'],
    'VUS-PR Rank (Multi.)': df_multi_sorted_ranks['VUS_PR_Rank']
}).sort_values(by='VUS-PR Rank (Uni.)', ascending=True)
save_combined_df(df_combined_ranks, 'ranks', rounding=1, font_size='footnotesize')

In [ ]:
%%capture
def spearman_correlation(df_scores: pd.DataFrame, model_name: str, mode: str, figsize: tuple = (8, 7), show: bool = False):
    """Calculate Spearman correlation between metrics and generate a heatmap.

    :param pd.DataFrame df_scores: DataFrame containing the scores for each model and dataset
    :param str model_name: Name of the model
    :param str mode: Type of time series data, defaults to 'uni'
    :param tuple figsize: Size of the heatmap figure, defaults to (8, 7)
    """
    # Drop columns if they exist
    df_scores = df_scores.drop(
        columns=['file', 'Time', 'HP', 'point_anomaly', 'seq_anomaly', 'num_anomaly'],
        errors='ignore'
    )

    # Spearman heatmap correlation between metrics
    df_corr = df_scores.corr(method='spearman')
    if mode in ['uni', 'multi']:
        df_corr.round(4).to_csv(f'{results_dir}/metrics/{mode}/spearman_{mode}_{model_name}.csv')
    else:
        df_corr.round(4).to_csv(f'{results_dir}/metrics/spearman_{mode}_{model_name}.csv')

    # Determine statistical significance of the Spearman correlations
    significance_matrix = df_scores.corr(method=lambda x, y: spearmanr(x, y)[1]) < 0.01
    # print(df_scores.corr(method=lambda x, y: spearmanr(x, y)[1]))

    # Plot the Spearman correlation heatmap for significant correlations
    plt.figure(figsize=figsize)

    sns.heatmap(
        df_corr, annot=True, fmt=".2f", cmap='Blues', cbar=False, square=True, vmin=0, vmax=1,
        # mask=~significance_matrix,
    )
    plt.grid(False)
    # plt.title(f'Spearman Correlation Heatmap ({mode.capitalize()}, {model_name})', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/spearman_{mode}_{model_name}_heatmap.png', dpi=400)
    if show:
        plt.show()

all_combined = []

for model_name, file_name in FILE_PATHS:
    df_scores_uni = pd.read_csv(f'{results_dir}/metrics/uni/{file_name}')
    spearman_correlation(df_scores_uni, model_name=model_name, mode='uni', figsize=(6, 6))

    df_scores_multi = pd.read_csv(f'{results_dir}/metrics/multi/{file_name}')
    spearman_correlation(df_scores_multi, model_name=model_name, mode='multi', figsize=(6, 6))

    df_scores_combined = pd.concat([df_scores_uni, df_scores_multi], axis=0)
    spearman_correlation(df_scores_combined, model_name=model_name, mode='combined', figsize=(6, 6))

    all_combined.append(df_scores_combined)

df_scores_combined_all = pd.concat(all_combined, axis=0)
spearman_correlation(df_scores_combined_all, model_name='all_models', mode='combined', figsize=(6, 6), show=True)

In [ ]:
%%capture
# Compare univariate and multivariate scores for models where both exist
def compare_uni_multi_scores_plot(df: pd.DataFrame, kind='bar', figsize=(9, 7)):
    """Plot comparing univariate and multivariate mean VUS-PR scores

    :param pd.DataFrame df: DataFrame containing mean VUS-PR scores for univariate and multivariate models
    :param str kind: Type of plot (e.g., 'bar', 'line'), defaults to 'bar'
    :param tuple figsize: Size of the figure (width, height)
    """
    plt.figure(figsize=figsize)

    kwargs_ = {
        'kind': kind,
        'x': 'Model',
        'y': ['Mean VUS-PR (Uni.)', 'Mean VUS-PR (Multi.)'],
        'ax': plt.gca(),
        'fontsize': 13,
    }
    if kind == 'bar':
        kwargs_ = {**kwargs_, 'edgecolor': 'black', 'linewidth': 0.5}
    elif kind == 'kde':
        kwargs_ = {**kwargs_, 'bw_method': 0.5, 'linewidth': 2}
    # else:
    #     raise ValueError("Invalid kind. Use 'bar' or 'kde'.")
    df.plot(**kwargs_)

    # Correlation between univariate and multivariate mean VUS-PR scores
    if kind == 'bar':
        spearman_corr, spearman_p_value = spearmanr(
            df['Mean VUS-PR (Uni.)'], df['Mean VUS-PR (Multi.)']
        )
        # Add Spearman correlation coefficient to plot
        plt.text(
            0.5, 0.95, f'Spearman correlation: {spearman_corr:.3f} (p-value: {spearman_p_value:.8f})',
            transform=plt.gca().transAxes, fontsize=13, ha='center', va='top',
            bbox=dict(facecolor='white', alpha=1.0, edgecolor='black'),
        )
        # Add Pearson correlation coefficient to plot
        pearson_corr, pearson_p_value = pearsonr(
            df['Mean VUS-PR (Uni.)'], df['Mean VUS-PR (Multi.)']
        )
        plt.text(
            0.5, 0.80, f'Pearson correlation: {pearson_corr:.3f} (p-value: {pearson_p_value:.8f})',
            transform=plt.gca().transAxes, fontsize=13, ha='center', va='top',
            bbox=dict(facecolor='white', alpha=1.0, edgecolor='black'),
        )

    # format axes and labels
    plt.xticks(rotation=90)
    plt.xlabel('Mean VUS-PR score')
    if kind == 'bar':
        plt.legend(title='Score Type')
        plt.xlabel('Model')
        plt.ylim(0, 1)
    elif kind == 'kde':
        plt.ylabel('Density')

    plt.tight_layout()
    plt.savefig(f'{plots_dir}/{kind}_uni_multi_mean_VUS-PR_comparison.png', dpi=500, bbox_inches='tight')
    plt.show()

# Create a DataFrame comparing mean VUS-PR scores for models that exist in both univariate and multivariate datasets
df_mode_comparison = pd.DataFrame({
    'Model': [model for model in models_uni if model in models_multi],
    'Mean VUS-PR (Uni.)': df_VUS_PR_uni[[model for model in models_uni if model in models_multi]].mean(),
    'Mean VUS-PR (Multi.)': df_VUS_PR_multi[[model for model in models_uni if model in models_multi]].mean()
}).sort_values(by='Mean VUS-PR (Uni.)', ascending=False)

print('Mean VUS-PR (Uni.):', df_mode_comparison['Mean VUS-PR (Uni.)'].mean())
print('Mean VUS-PR (Multi.):', df_mode_comparison['Mean VUS-PR (Multi.)'].mean())

# Draw plots
# compare_uni_multi_scores_plot(df_mode_comparison, kind='hist', figsize=(6, 4))
compare_uni_multi_scores_plot(df_mode_comparison, kind='kde', figsize=(8, 5))
compare_uni_multi_scores_plot(df_mode_comparison, kind='bar', figsize=(9, 7))


In [ ]:
%%capture
def plot_difference_bar(df: pd.DataFrame, target_col: str, figsize: tuple = (9, 6), metric: str = 'Mean VUS-PR'):
    """Bar plot of the difference between multivariate and univariate mean VUS-PR scores for each model.

    :param pd.DataFrame df: DataFrame containing the scores for each model
    :param str target_col: Column name for the target values to plot
    :param tuple figsize: Figure size for the plot, defaults to (9, 6)
    :param str metric: Metric name for labeling, defaults to 'Mean VUS-PR'
    """
    # Sort values by the target column in descending order
    df = df.sort_values(by=target_col, ascending=False)

    # Color the bars based on whether the difference is positive or negative
    df['Color'] = df[target_col].apply(lambda x: 'lightgreen' if x > 0 else 'pink')

    # Draw plot
    plt.figure(figsize=figsize)
    plt.bar(
        df['Model'], df[target_col], width=0.55, edgecolor='black', linewidth=0.5, color=df['Color']
    )
    # Format axes, labels and grid lines
    plt.axhline(0, color='red', linestyle='--', linewidth=1)
    plt.xticks(rotation=90)
    plt.ylabel(f'{target_col} in {metric} (Multi. - Uni.)')
    plt.grid(False, which='both', axis='x')

    # Set limits
    limit = max(abs(df[target_col].min()), abs(df[target_col].max())) * 1.4
    plt.ylim(-limit, limit)
    label_offset = 0.01 * limit

    # Show values on top of the bars
    for index, value in enumerate(df[target_col]):
        y = value + (label_offset if value > 0 else -label_offset)
        if 'Difference (%)' in target_col:
            text = f'{value:.2f}%'
        else:
            text = f'{value:.4f}'
        alignment = 'bottom' if value > 0 else 'top'
        rotation = 90
        plt.text(x=index, y=y, s=text, ha='center', va=alignment, fontsize=11, rotation=rotation)

    # Highlight model names in bold
    for label in plt.gca().get_xticklabels():
        if label.get_text() in NEW_MODELS:
            label.set_fontweight('bold')

    # Make the box outlines black for the highlighted models
    for i, patch in enumerate(plt.gca().patches):
        if df.iloc[i]['Model'] in NEW_MODELS:
            patch.set_edgecolor('black')
            patch.set_linewidth(1)
            # Whiskers, caps, median, etc.
            for line in plt.gca().lines[i * 6:(i + 1) * 6]:
                line.set_color('black')
                line.set_linewidth(1)

    # Save the plot and show
    plt.tight_layout()
    file_id = 'percentage' if 'Difference (%)' in target_col else 'absolute'
    plt.savefig(f'{plots_dir}/bar_diff_{metric}_{file_id}.png', dpi=500, bbox_inches='tight')
    plt.show()

# Plot difference between univariate and multivariate mean VUS-PR scores per model
df_mode_comparison['Difference'] = df_mode_comparison['Mean VUS-PR (Multi.)'] - df_mode_comparison['Mean VUS-PR (Uni.)']
plot_difference_bar(df_mode_comparison, target_col='Difference')

# Same plot but calculate the difference as a percentage
df_mode_comparison['Difference (%)'] = (df_mode_comparison['Difference'] / df_mode_comparison['Mean VUS-PR (Uni.)']) * 100
plot_difference_bar(df_mode_comparison, target_col='Difference (%)')

## Formatting CSV Results by Model

In [ ]:
files = [('LUNAR', 'LunarADModel.csv'), ('PyCaret', 'PyCaretADModel.csv'), ('PyOD', 'TimeSeriesODModel.csv')]
all_results = {}

# Iterate over the files and modes to read CSV files and calculate mean scores
for model_name, file_name in files:

    for mode in ['uni', 'multi']:
        # Check if the CSV file exists before reading
        csv_path = Path(f'{results_dir}/metrics/{mode}/{file_name}')
        if not csv_path.exists():
            print(f"File {csv_path} does not exist. Skipping...")
            continue
        # Read file, drop columns, calculate mean scores
        df_scores = pd.read_csv(csv_path).drop(
            columns=['file', 'Time', 'HP', 'point_anomaly', 'seq_anomaly', 'num_anomaly'], errors='ignore'
        )
        all_results[(model_name, mode)] = df_scores.mean()

# Create DataFrame and save to CSV
df_all_results = pd.DataFrame(all_results).round(3)
csv_path = Path(f'{results_dir}/metrics/all_models_mean_scores.csv')
df_all_results.T.to_csv(csv_path)

# Save as LaTeX table
df_all_results.to_latex(
    csv_path.with_suffix('.tex'), index=True, header=True, float_format="%.3f",
    caption='Mean anomaly detection scores for all models across univariate and multivariate datasets.',
    label='tab:ch5:all_scores_mean_new_models',
    column_format='l' + 'c' * (len(df_all_results.columns) + 1),
)
# Add "\centering" to the LaTeX table
with open(csv_path.with_suffix('.tex'), 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open(csv_path.with_suffix('.tex'), 'w', encoding='utf-8') as f:
    for line in lines:
        if line.strip() == '\\begin{table}':
            f.write('\\begin{table}[htbp]\n')
            f.write('\\centering\n')
        else:
            f.write(line)